# STEP 08 — 2b: 방문인구 join
**2부A-2a 완료 후 실행**

In [ ]:
import csv, json, math, os, warnings
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, roc_auc_score)
from sklearn.model_selection import train_test_split
import geopandas as gpd
from shapely.geometry import Point

try:
    import statsmodels.api as sm
    HAS_STATSMODELS = True
except ImportError:
    from sklearn.linear_model import PoissonRegressor
    HAS_STATSMODELS = False

warnings.filterwarnings("ignore", category=UserWarning)

# ── 경로 설정 ──────────────────────────────────────────────────────────────────
# COMPAS 구조: 노트북과 같은 위치에 data/ 폴더가 있음
BASE_DIR   = os.getcwd()          # 노트북 실행 위치 (data/ 폴더의 부모)
EPDO_DIR   = os.getcwd()          # 분석 결과 저장 위치
OUTPUT_DIR = os.path.join(EPDO_DIR, "epdo_analysis", "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"BASE_DIR  : {BASE_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

In [ ]:
import csv
import json
import os
import warnings
from collections import defaultdict, Counter

import geopandas as gpd
from shapely.geometry import Point

warnings.filterwarnings("ignore", category=UserWarning)   # CRS 관련 경고 억제

# 입력
GAP_FILE     = os.path.join(EPDO_DIR, "epdo_analysis", "output", "06_인프라공백_고위험격자.csv")
STEP07_FILE  = os.path.join(EPDO_DIR, "epdo_analysis", "output", "07_링크_속도혼잡_보강.csv")
GRID_FILE    = os.path.join(BASE_DIR, "data", "01._격자_(4개_시·구).geojson")
RES_FILE     = os.path.join(BASE_DIR, "data", "03._성연령별_거주인구(격자).csv")
FLOAT_FILE   = os.path.join(BASE_DIR, "data", "04._성연령별_유동인구.csv")
WORK_FILE    = os.path.join(BASE_DIR, "data", "05._시간대별_직장인구.csv")
VISIT_FILE   = os.path.join(BASE_DIR, "data", "06._시간대별_방문인구.csv")
SVC_FILE     = os.path.join(BASE_DIR, "data", "07._주중주말_서비스인구.csv")
ROAD_FILE    = os.path.join(BASE_DIR, "data", "08.상세도로망_네트워크.geojson")
LU_FILE      = os.path.join(BASE_DIR, "data", "22._토지이용계획도_(4개_신도시).geojson")
LU_HANAM     = os.path.join(BASE_DIR, "data", "23._토지이용계획도_(하남교산).geojson")

# 출력
OUTPUT_PATH  = os.path.join(EPDO_DIR, "epdo_analysis", "output", "08_격자_종합위험지수.csv")
HANAM_OUTPUT = os.path.join(EPDO_DIR, "epdo_analysis", "output", "08_하남교산_예측위험도.csv")

CRS_PROJ     = "EPSG:5186"   # 한국 중부원점 (단위: 미터, centroid 계산 정확)
CRS_GEO      = "EPSG:4326"   # WGS84

# 교통약자 취약 시간대: 등교(07~09), 하교(13~16), 퇴근(17~19)
VULN_SLOTS   = {7, 8, 9, 13, 14, 15, 16, 17, 18, 19}
SLOT_COLS    = [f"TMST_{str(h).zfill(2)}" for h in VULN_SLOTS]
ALL_SLOTS    = [f"TMST_{str(h).zfill(2)}" for h in range(24)]

# 토지이용 → 위험 카테고리 매핑
SCHOOL_TYPES    = {"학교", "초등학교", "중학교", "고등학교", "교육시설"}
RESIDENT_TYPES  = {"아파트", "연립주택", "다세대주택", "단독주택", "공동주택", "주상복합", "연립"}
COMMERCIAL_TYPES = {"상업", "일반상업", "근린상업", "근린생활시설용지", "상업시설"}


# ── 유틸 ────────────────────────────────────────────────────────────────────

def load_csv_dict(path, key_col):
    with open(path, "r", encoding="utf-8-sig") as f:
        return {row[key_col]: row for row in csv.DictReader(f)}


def safe_float(val, default=0.0):
    try:
        return float(val or 0)
    except (ValueError, TypeError):
        return default


def points_to_grid(lon_col, lat_col, rows, value_cols, grid_gdf):
    """lon/lat 포인트 → 격자 공간 join 후 격자별 집계 반환"""
    pts = []
    for row in rows:
        try:
            lon, lat = float(row[lon_col]), float(row[lat_col])
        except (ValueError, KeyError):
            continue
        rec = {"geometry": Point(lon, lat)}
        for c in value_cols:
            rec[c] = safe_float(row.get(c))
        pts.append(rec)

    pts_gdf  = gpd.GeoDataFrame(pts, crs=CRS_GEO)
    joined   = gpd.sjoin(pts_gdf, grid_gdf[["gid", "geometry"]], how="left", predicate="within")
    return joined


def median(values):
    sv = sorted(v for v in values if v is not None)
    if not sv:
        return 0
    mid = len(sv) // 2
    return (sv[mid - 1] + sv[mid]) / 2 if len(sv) % 2 == 0 else sv[mid]


# ── 메인 ─────────────────────────────────────────────────────────────────────
import gc


## ── 1. 고위험 격자 목록

In [ ]:
print("\n[1] 고위험 격자 로드 중...")
gap_data = load_csv_dict(GAP_FILE, "grid_gid")
print(f"    고위험 격자 수: {len(gap_data):,}개")

## ── 2. 격자 GeoDataFrame (고위험만 필터)

In [ ]:
print("\n[2] 격자 폴리곤 로드 중...")
grid_gdf = gpd.read_file(GRID_FILE)
grid_gdf = grid_gdf[grid_gdf["gid"].isin(gap_data.keys())].reset_index(drop=True)
print(f"    고위험 격자 폴리곤: {len(grid_gdf):,}개")

## ── 7. 06번 방문인구 — 취약 시간대 공간 join

In [ ]:
print("[7] 방문인구(취약 시간대) 공간 join 중... (청크 처리)")
from collections import defaultdict as _dd
from shapely.geometry import Point as _Pt
import gc

CHUNK = 30000
_visit_raw = _dd(lambda: {"vuln_visit": 0.0, "total_visit": 0.0})

def _proc_visit_chunk(rows, agg):
    pts = []
    for row in rows:
        try: lon, lat = float(row["lon"]), float(row["lat"])
        except: continue
        rec = {"geometry": _Pt(lon, lat)}
        for c in SLOT_COLS + ALL_SLOTS:
            rec[c] = safe_float(row.get(c))
        pts.append(rec)
    if not pts: return
    _gdf = gpd.GeoDataFrame(pts, crs=CRS_GEO)
    _j   = gpd.sjoin(_gdf, grid_gdf[["gid","geometry"]], how="left", predicate="within")
    for gid, grp in _j.dropna(subset=["gid"]).groupby("gid"):
        agg[str(gid)]["vuln_visit"]  += sum(grp[c].sum() for c in SLOT_COLS if c in grp.columns)
        agg[str(gid)]["total_visit"] += sum(grp[c].sum() for c in ALL_SLOTS if c in grp.columns)
    del _gdf, _j
    gc.collect()

with open(VISIT_FILE, "r", encoding="utf-8-sig") as _f:
    _reader = csv.DictReader(_f)
    _chunk = []
    for _row in _reader:
        _chunk.append(_row)
        if len(_chunk) >= CHUNK:
            _proc_visit_chunk(_chunk, _visit_raw)
            _chunk = []
    if _chunk:
        _proc_visit_chunk(_chunk, _visit_raw)

visit_agg = {gid: {"vuln_visit": round(v["vuln_visit"],4), "total_visit": round(v["total_visit"],4)}
             for gid, v in _visit_raw.items()}
print(f"    방문인구 매칭 격자: {len(visit_agg):,}개")
gc.collect()


In [ ]:
import json as _json
_tmp = os.path.join(EPDO_DIR, "epdo_analysis", "output")

with open(os.path.join(_tmp, "_step08_visit_agg.json"), "w", encoding="utf-8") as _f:
    _json.dump(visit_agg, _f, ensure_ascii=False)
print("저장 완료 → 2부A-2c 실행")